# Custom Data Training (Phase 6)

This notebook demonstrates how to train the CNN model on custom OHLCV data from CSV files.

**Features Covered:**
- Loading data from CSV files
- Using pre-configured presets for different asset classes
- Configuring timeframes and window sizes
- Comparing sample modes (overlapping vs non-overlapping)
- Training on multiple data sources

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Import Phase 6 modules
from src.data.csv_loader import load_csv, get_data_summary, estimate_samples, infer_timeframe
from src.data.data_source import DataSourceFactory, CSVSource, YahooFinanceSource, quick_train_data
from src.data.preprocessor import compare_sample_modes, estimate_sample_count
from src.utils.presets import list_presets, get_preset, describe_preset, print_all_presets
from src.utils.config import print_config, get_config_dict, SampleMode, Timeframe

print("Phase 6 modules loaded successfully!")

## 1. Available Presets

Phase 6 includes pre-configured presets for different asset classes and timeframes.

In [ ]:
# View all available presets
print_all_presets()

In [ ]:
# Get details about a specific preset
print(describe_preset("crypto_hourly"))

In [ ]:
# Get preset configuration for use in training
crypto_config = get_preset("crypto_hourly").get_config()
print("Crypto Hourly Config:")
for k, v in crypto_config.items():
    print(f"  {k}: {v}")

## 2. Loading Data from CSV

The CSV loader supports flexible column mapping and automatic format detection.

In [ ]:
# Example: Create sample CSV data for demonstration
# In practice, you would load your own CSV file

# Generate synthetic hourly data for demonstration
np.random.seed(42)
dates = pd.date_range(start='2023-01-01', periods=2000, freq='h')

# Generate realistic OHLCV data
base_price = 100
returns = np.random.randn(2000) * 0.01
prices = base_price * np.exp(np.cumsum(returns))

sample_data = pd.DataFrame({
    'datetime': dates,
    'open': prices * (1 + np.random.randn(2000) * 0.001),
    'high': prices * (1 + np.abs(np.random.randn(2000)) * 0.005),
    'low': prices * (1 - np.abs(np.random.randn(2000)) * 0.005),
    'close': prices,
    'volume': np.random.randint(1000, 10000, 2000)
})

# Save to CSV (simulating a real data file)
csv_path = Path('../data/custom/sample_hourly_data.csv')
csv_path.parent.mkdir(parents=True, exist_ok=True)
sample_data.to_csv(csv_path, index=False)

print(f"Created sample data: {csv_path}")
print(f"Rows: {len(sample_data)}")
sample_data.head()

In [ ]:
# Load the CSV file with automatic column detection
df = load_csv(csv_path)

# Get data summary
summary = get_data_summary(df)
print("\nData Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

In [ ]:
# Visualize the loaded data
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Price chart
axes[0].plot(df.index, df['Close'], 'b-', linewidth=0.5)
axes[0].fill_between(df.index, df['Low'], df['High'], alpha=0.2)
axes[0].set_title('Price Data')
axes[0].set_ylabel('Price')

# Volume chart
axes[1].bar(df.index, df['Volume'], width=0.02, alpha=0.7)
axes[1].set_title('Volume')
axes[1].set_ylabel('Volume')

plt.tight_layout()
plt.show()

## 3. Understanding Sample Modes

Phase 6 introduces different sample generation modes to address the correlation problem in overlapping windows.

In [ ]:
# Compare different sample modes for our data
preset = get_preset("crypto_hourly")

comparison = compare_sample_modes(
    df,
    window_size=preset.window_size,
    horizon=preset.horizons[0],
    custom_stride=10
)

print("Sample Mode Comparison:")
print("=" * 60)
for mode_name, stats in comparison.items():
    print(f"\n{mode_name.upper()}:")
    print(f"  Description: {stats['description']}")
    print(f"  Total samples: {stats['samples']}")
    print(f"  Training samples: {stats['train_samples']}")
    print(f"  Sufficient data: {'Yes' if stats['sufficient'] else 'No'}")

In [ ]:
# Visualize sample coverage for different modes
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

window_size = 168
horizon = 6
data_len = 500  # Show first 500 points

for ax, (mode_name, stride) in zip(axes, [
    ('Overlapping (stride=1)', 1),
    ('Strided (stride=10)', 10),
    ('Non-overlapping', window_size + horizon)
]):
    ax.plot(range(data_len), df['Close'].iloc[:data_len].values, 'b-', alpha=0.5, linewidth=0.5)
    
    # Show window coverage
    for i in range(0, data_len - window_size - horizon, stride):
        if i + window_size + horizon <= data_len:
            ax.axvspan(i, i + window_size, alpha=0.1, color='green')
            ax.axvline(i + window_size + horizon, color='red', alpha=0.3, linewidth=0.5)
        if i > 3 * (window_size + horizon):  # Limit visualization
            break
    
    ax.set_title(mode_name)
    ax.set_ylabel('Price')

axes[-1].set_xlabel('Time Index')
plt.tight_layout()
plt.show()

## 4. Using the Unified Data Interface

The `DataSource` abstraction provides a consistent interface for loading data from different sources.

In [ ]:
# Create data source from CSV
csv_source = DataSourceFactory.create_csv(csv_path)

# Get source summary
print("CSV Source Summary:")
for k, v in csv_source.get_summary().items():
    print(f"  {k}: {v}")

In [ ]:
# Estimate samples before training
preset = get_preset("crypto_hourly")

estimate = csv_source.estimate_samples(
    window_size=preset.window_size,
    horizon=preset.horizons[0],
    stride=1  # Overlapping mode
)

print("Sample Estimate (Overlapping):")
for k, v in estimate.items():
    print(f"  {k}: {v}")

In [ ]:
# Prepare training data
X_train, y_train, X_val, y_val, X_test, y_test = csv_source.prepare_training_data(
    window_size=preset.window_size,
    horizon=preset.horizons[0],
    stride=1,  # Overlapping
    val_ratio=0.15,
    test_ratio=0.15
)

print(f"\nTraining Data Prepared:")
print(f"  X_train shape: {X_train.shape}")
print(f"  X_val shape: {X_val.shape}")
print(f"  X_test shape: {X_test.shape}")
print(f"  Bullish ratio (train): {y_train.mean():.2%}")

## 5. Training with Custom Data

Now let's train the model using our custom data.

In [ ]:
from src.models.cnn import StockCNN
from src.data.dataset import create_dataloaders
from src.training.trainer import Trainer, create_trainer
from src.utils.config import DEVICE

# Create dataloaders
train_loader, val_loader = create_dataloaders(
    X_train, y_train, X_val, y_val,
    batch_size=64  # Smaller batch for demo
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

In [ ]:
# Create model with correct window size
model = StockCNN(
    window_size=preset.window_size,
    num_channels=5  # OHLCV
)

print(model.summary())

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    learning_rate=1e-3,
    device=DEVICE
)

print(f"Training on: {DEVICE}")

In [ ]:
# Train for a few epochs (demo)
# For real training, increase num_epochs
metrics = trainer.train(
    num_epochs=5,  # Use more epochs for real training
    early_stopping_patience=3
)

In [ ]:
# Plot training history
from src.visualization.plots import plot_training_history

plot_training_history(metrics)

## 6. Comparing Sample Modes (Training Experiment)

Let's compare training with different sample modes to understand their impact.

In [ ]:
# This cell demonstrates how to compare different sample modes
# Uncomment to run (takes longer)

# def train_with_mode(source, preset, stride, mode_name, epochs=10):
#     """Train model with specific sample mode."""
#     X_train, y_train, X_val, y_val, X_test, y_test = source.prepare_training_data(
#         window_size=preset.window_size,
#         horizon=preset.horizons[0],
#         stride=stride
#     )
#     
#     if len(X_train) < 50:
#         print(f"Skipping {mode_name}: insufficient samples ({len(X_train)})")
#         return None
#     
#     train_loader, val_loader = create_dataloaders(X_train, y_train, X_val, y_val, batch_size=32)
#     model = StockCNN(window_size=preset.window_size)
#     trainer = Trainer(model, train_loader, val_loader)
#     
#     print(f"\nTraining {mode_name} (stride={stride}, samples={len(X_train)})...")
#     metrics = trainer.train(num_epochs=epochs, early_stopping_patience=5)
#     
#     return {
#         'mode': mode_name,
#         'stride': stride,
#         'train_samples': len(X_train),
#         'best_val_loss': metrics.get_best_metrics()['best_val_loss'],
#         'best_val_accuracy': metrics.get_best_metrics()['best_val_accuracy'],
#     }
# 
# # Compare modes
# results = []
# for mode_name, stride in [('overlapping', 1), ('strided', 10)]:
#     result = train_with_mode(csv_source, preset, stride, mode_name)
#     if result:
#         results.append(result)
# 
# # Display comparison
# if results:
#     comparison_df = pd.DataFrame(results)
#     print("\nSample Mode Comparison Results:")
#     print(comparison_df.to_string(index=False))

print("Sample mode comparison code available above (uncomment to run)")

## 7. Using Yahoo Finance Data with New Features

The unified interface also works with Yahoo Finance data.

In [ ]:
# Create Yahoo Finance data source
yahoo_source = DataSourceFactory.create_yahoo(
    ticker="AAPL",
    start_date="2020-01-01",
    end_date="2024-01-01"
)

# Get summary
print("Yahoo Finance Source Summary:")
for k, v in yahoo_source.get_summary().items():
    print(f"  {k}: {v}")

In [ ]:
# Estimate samples for stock daily preset
stock_preset = get_preset("stock_daily")

estimate = yahoo_source.estimate_samples(
    window_size=stock_preset.window_size,
    horizon=stock_preset.horizons[0]
)

print("Sample Estimate (Stock Daily):")
for k, v in estimate.items():
    print(f"  {k}: {v}")

## 8. Quick Training Helper

For convenience, use the `quick_train_data` function to load data in one step.

In [ ]:
# Quick load from ticker
X_train, y_train, X_val, y_val, X_test, y_test = quick_train_data(
    "MSFT",  # Ticker symbol
    window_size=256,
    horizon=5,
    stride=1,
    start_date="2019-01-01",
    end_date="2024-01-01"
)

print(f"Loaded MSFT data:")
print(f"  Train: {X_train.shape}")
print(f"  Val: {X_val.shape}")
print(f"  Test: {X_test.shape}")

In [ ]:
# Quick load from CSV
X_train, y_train, X_val, y_val, X_test, y_test = quick_train_data(
    csv_path,  # CSV file path
    window_size=168,
    horizon=6,
    stride=5  # Strided mode
)

print(f"Loaded CSV data (strided):")
print(f"  Train: {X_train.shape}")
print(f"  Val: {X_val.shape}")
print(f"  Test: {X_test.shape}")

## 9. Summary

Phase 6 adds powerful features for custom data:

1. **CSV Loading**: Flexible loader with auto-detection of columns and dates
2. **Presets**: Pre-configured settings for different asset classes
3. **Sample Modes**: Control sample overlap for better training
4. **Unified Interface**: Consistent API for different data sources
5. **Quick Helpers**: One-line data loading functions

Use the configuration that best matches your data characteristics!

In [ ]:
# Final configuration summary
print_config()